# Phase 5 — Main experiments (RQ1 and RQ2)
**Goal:** run the full experiment and produce the main results table.

**The grid**
- 15 datasets × 6 models × 13 noise conditions × **R = 51** rounds ≈ **59,700 model trainings**
- 13 conditions = clean (0 %) + 3 noise types (symmetric, one-sided, boundary) × 4 rates (5, 10, 20, 30 %)

**Research questions answered here**
- **RQ1:** How much does each model's error grow as label noise increases?
- **RQ2:** Is that extra error mainly **bias** or **variance**, and does it depend on the noise type and the model family?

This notebook gives a **first look** using averages. The proper statistical tests and the paper's figures come in **Phase 7**.

Kernel: **Python (labelnoise)**.

## Step 1 — How the experiment is run
The work is done by the script **`src/run_main.py`**, not by this notebook, because it runs for over an hour. It was started from a terminal:
```
conda run -n labelnoise python src/run_main.py
```
**Three design choices make it safe to run for a long time:**
1. **Saves after every condition.** Each finished condition is appended as one row to `results/raw/main/<dataset>.csv`. A crash, a closed laptop or a power cut loses at most one condition.
2. **Resumable.** On start, it reads what is already saved and skips it. Running it again simply continues.
3. **Keeps the raw predictions.** Each condition's prediction matrix P (51 rounds × test points) is saved in `results/raw/main_preds/`. Later analyses (e.g. "is the damage concentrated near the boundary?" in Phase 6) can reuse them without re-training anything.

The datasets run from smallest to largest, so early results appear within minutes.

The cell below calls the same function. Because it is resumable, it finishes instantly when everything is already done, and fills in anything missing.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "src"))
from core import MODEL_NAMES, MODEL_FAMILY, NOISE_TYPES
import run_main

run_main.run_all()          # skips every condition that is already saved

## Step 2 — Merge the results and check them
We combine the 15 per-dataset files into one table, `results/main_results.csv`, and check it before looking at any numbers:

| Check | Expected |
|---|---|
| Rows | 15 × 6 × 13 = **1,170**, no duplicates |
| Missing values | none |
| Rounds | R = 51 on every row |
| Identity | error = bias + unbiased var − biased var on every row |
| Noise rate | realised flip rate matches the target (one-sided: target × class-1 share) |

In [ ]:
OUT = BASE / "results"
res = pd.concat([pd.read_csv(f) for f in sorted((OUT / "raw" / "main").glob("*.csv"))], ignore_index=True)
res.to_csv(OUT / "main_results.csv", index=False)

expected_flip = np.where(res.noise_type == "one_sided", res.rho * res.pos_share_clean, res.rho)
checks = {
    "rows = 1,170": len(res) == 15 * 6 * 13,
    "no duplicate conditions": not res.duplicated(["dataset", "model", "noise_type", "rho"]).any(),
    "no missing values": not res.isna().any().any(),
    "R = 51 everywhere": (res.R == 51).all(),
    "identity holds": res.identity_gap.abs().max() < 1e-12,
    "flip rate on target (±0.01)": np.abs(res.flip_rate - expected_flip).max() < 0.01,
}
for k, ok in checks.items():
    print(f"{'OK  ' if ok else 'FAIL'}  {k}")
print(f"\n{len(res)} rows, {res.dataset.nunique()} datasets; total compute {res.seconds.sum() / 60:.0f} min")

## Step 3 — Measure the damage caused by noise
The numbers we care about are **changes relative to clean labels**, for the same dataset and model:

- **Δ error** = error(noisy) − error(clean): how much worse the model got
- **Δ bias** = bias(noisy) − bias(clean): damage from learning a *wrong rule*
- **Δ net var** = net variance(noisy) − net variance(clean): damage from *instability*

Because error = bias + net variance, **Δ error = Δ bias + Δ net var**. The two parts add up exactly to the total damage.

In [ ]:
clean = res[res.noise_type == "none"].set_index(["dataset", "model"])
noisy = res[res.noise_type != "none"].copy()
key = list(zip(noisy.dataset, noisy.model))
for col in ["error", "bias", "net_var"]:
    noisy["d_" + col] = noisy[col].to_numpy() - clean.loc[key, col].to_numpy()
noisy["clean_error"] = clean.loc[key, "error"].to_numpy()
noisy.to_csv(OUT / "main_results_deltas.csv", index=False)
assert np.allclose(noisy.d_error, noisy.d_bias + noisy.d_net_var)
print("Δ error = Δ bias + Δ net var on every row.")

## Step 4 — First look at RQ1: how much does noise hurt each model?
Average **Δ error** over the 15 datasets, for each model, noise type and rate. The values are in **percentage points** (pp): +5.0 means the error rate rose by 5 points, e.g. from 10 % to 15 %.

**How to read it:** look down a column to compare models at the same noise; look along a row to see how damage grows with the rate.

In [ ]:
TYPES = ["symmetric", "one_sided", "boundary"]
rq1 = (noisy.pivot_table(index="model", columns=["noise_type", "rho"], values="d_error", aggfunc="mean")
            .reindex(MODEL_NAMES)[TYPES] * 100).round(1)
rq1.to_csv(OUT / "tables" / "phase5_rq1_delta_error_pp.csv")
rq1

**Same table, by family** (single tree vs parallel ensembles vs sequential ensembles), at 20 % and 30 % noise:

In [ ]:
fam = (noisy[noisy.rho.isin([0.2, 0.3])]
       .pivot_table(index="family", columns=["noise_type", "rho"], values="d_error", aggfunc="mean")
       .reindex(["single", "parallel", "sequential"])[TYPES] * 100).round(1)
fam

## Step 5 — First look at RQ2: is the damage bias or variance?
This is the core question of the paper. At **20 % noise**, we split each model's average damage into its two parts and compute the **variance share**:

  variance share = Δ net var / Δ error

- close to **100 %**: the damage is mostly *instability*, so the fix is variance reduction (more averaging, early stopping)
- close to **0 %**: the damage is mostly a *wrong rule*, so the fix is cleaning or relabelling the data
- **above 100 %** or **below 0 %**: the two parts pull in opposite directions (e.g. bias went *down* while variance went up)

We compute the share from the *averages* over datasets, not as an average of per-dataset shares. A dataset where Δ error is almost zero would otherwise produce a huge, meaningless ratio.

In [ ]:
def split(rho):
    g = noisy[noisy.rho == rho].groupby(["noise_type", "model"])[["d_error", "d_bias", "d_net_var"]].mean()
    g["variance share %"] = 100 * g.d_net_var / g.d_error
    g[["d_error", "d_bias", "d_net_var"]] *= 100
    g = g.rename(columns={"d_error": "Δ error (pp)", "d_bias": "Δ bias (pp)", "d_net_var": "Δ net var (pp)"})
    return g.reindex(pd.MultiIndex.from_product([TYPES, MODEL_NAMES], names=["noise_type", "model"])).round(1)

rq2 = split(0.20)
rq2.to_csv(OUT / "tables" / "phase5_rq2_split_20pct.csv")
rq2

**The same split by family and noise type, at every rate.** This shows whether the pattern is stable as noise grows:

In [ ]:
g = noisy.groupby(["noise_type", "family", "rho"])[["d_error", "d_net_var"]].mean()
share = (100 * g.d_net_var / g.d_error).unstack("rho").round(0)
share = share.reindex(pd.MultiIndex.from_product([TYPES, ["single", "parallel", "sequential"]]))
share.columns = [f"{int(r * 100)} %" for r in share.columns]
share.to_csv(OUT / "tables" / "phase5_variance_share_by_family.csv")
print("Variance share of the damage (%), by noise type, family and noise rate")
share

## Step 6 — Side effect: one-sided noise changes the class balance
One-sided noise only turns positives into negatives, so the training data looks **more imbalanced** than reality. That alone can push models towards predicting class 0, which is a *bias* effect. We record how large the shift is:

In [ ]:
os_ = noisy[noisy.noise_type == "one_sided"].groupby(["dataset", "rho"]).agg(
    clean=("pos_share_clean", "first"), noisy=("pos_share_noisy", "mean")).reset_index()
bal = os_.pivot(index="dataset", columns="rho", values="noisy")
bal.insert(0, "clean", os_.groupby("dataset").clean.first())
bal.columns = ["clean"] + [f"{int(r * 100)} % one-sided" for r in bal.columns[1:]]
print("Class-1 share in the training labels")
bal.round(3).sort_values("clean")

## What to take from this first look (and what not to)
- These are **averages over 15 datasets**. They show the overall pattern but say nothing yet about whether differences are **statistically reliable**. Phase 7 adds the Friedman / Nemenyi and Wilcoxon tests and the paper's figures.
- Compare what you see with the hypotheses (H1: symmetric noise → mostly variance; H2: one-sided → mostly bias; H4: sequential models hurt more than parallel ones). Write down, in your own words, which look supported, which do not, and anything surprising. Those notes become the start of the *Results* section.

## Phase 5 checklist
- [ ] All Step 2 checks print **OK**
- [ ] `results/main_results.csv` (1,170 rows) and `results/main_results_deltas.csv` saved
- [ ] First-look tables saved in `results/tables/phase5_*.csv`
- [ ] Your own notes on H1, H2 and H4 written

**Next: Phase 6**, the extra experiments: where the variance comes from (RQ3) and whether regularisation helps (RQ4).